In [1]:
import winogender_contextuality as wc
from winogender_contextuality.modeling.ModelProbs import *
from winogender_contextuality.modeling.contextuality import *

2025-07-28 19:42:04.848 | INFO     | winogender_contextuality.config:<module>:13 - PROJ_ROOT path is: /home/sagar/winogender_contextuality
2025-07-28 19:42:04.849 | INFO     | winogender_contextuality.config:<module>:17 - DATA_ROOT path is: /data_users1/sagar/winogender_contextuality


2025-07-28 19:42:10.425 | INFO     | winogender_contextuality.modeling.run_local:<module>:9 - torch available: True
2025-07-28 19:42:10.426 | INFO     | winogender_contextuality.modeling.run_local:<module>:10 - 12.6
2025-07-28 19:42:10.468 | INFO     | winogender_contextuality.modeling.run_local:<module>:12 - _CudaDeviceProperties(name='NVIDIA TITAN Xp', major=6, minor=1, total_memory=12188MB, multi_processor_count=30, uuid=69e3ceae-9a8c-607c-5cd7-cde99d6659aa, L2_cache_size=3MB)


In [2]:
from pathlib import Path
import os
from matplotlib import pyplot as plt
import pandas as pd
import ast
import importlib
import torch.nn.functional as F
#import torch

## Loading Model

In [3]:
HF_KEY = ""

In [4]:
wc.config.MODELS_DIR

PosixPath('/data_users1/sagar/winogender_contextuality/models')

In [5]:
mp = wc.modeling.ModelProbs.ModelProbs(mode='gpu', 
                                       model_name="meta-llama/Llama-3.2-1B-Instruct", 
                                       key=HF_KEY, 
                                       model_path=wc.config.MODELS_DIR)

In [6]:
mp.load_model()

2025-07-28 19:42:16.807 | INFO     | winogender_contextuality.modeling.run_local:load_model:33 - Connected with Huggingface
2025-07-28 19:42:16.807 | INFO     | winogender_contextuality.modeling.run_local:load_model:43 - Loading quantized model
2025-07-28 19:42:20.351 | INFO     | winogender_contextuality.modeling.run_local:load_model:50 - Model cached in /data_users1/sagar/winogender_contextuality/models/cache
2025-07-28 19:42:20.352 | INFO     | winogender_contextuality.modeling.ModelProbs:load_model:55 - Loaded model meta-llama/Llama-3.2-1B-Instruct to /data_users1/sagar/winogender_contextuality/models


## Loading Data

In [7]:
from winogender_contextuality.modeling.prompting import *
from winogender_contextuality.utils import *
from winogender_contextuality.modeling.pipeline import * 
from itertools import chain

In [8]:
pairs = pd.read_csv(wc.config.PROCESSED_DATA_DIR / "wino_pairs.tsv", sep="\t")

In [9]:
pairs.head()

,template_1,differences_1,referent_1,template_2,differences_2,referent_2
0,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
1,The customer met with the technician to get he...,"['his', 'her']",customer,The customer met with the technician to find o...,"['he', 'she']",technician
2,The technician gave the customer feedback on B...,"['his', 'her']",customer,The customer gave the technician feedback on B...,"['his', 'her']",technician
3,The technician informed the customer that BLAN...,"['he', 'she']",customer,The technician informed the customer that BLAN...,"['he', 'she']",technician
4,The technician told the customer that BLANK sh...,"['he', 'she']",customer,The technician told the customer that BLANK wa...,"['he', 'she']",technician


# Test Prompts

In [10]:
test_prompt = wc.modeling.prompting.get_role_content_prompt(game=False, options=pairs.differences_1[0], 
                                      sentence=pairs.template_1[0])

In [11]:
test_logits = mp.get_raw_logits(prompt=test_prompt).to('cpu')

In [12]:
mp.get_token_ids(options=[" "+s for s in ast.literal_eval(pairs.differences_1[0])])

[[568], [1364]]

In [13]:
[" "+s for s in ast.literal_eval(pairs.differences_1[0])]

[' he', ' she']

In [14]:
test_logits[568], test_logits[1364]

(tensor(20.8281, dtype=torch.float16), tensor(21.2656, dtype=torch.float16))

In [15]:
test_prompt_2 = wc.modeling.prompting.get_role_content_prompt(game=False, options=pairs.differences_2[0], 
                                      sentence=pairs.template_2[0])
test_logits_2 = mp.get_raw_logits(prompt=test_prompt_2).to('cpu')
test_logits_2[568], test_logits_2[1364]

(tensor(19.7031, dtype=torch.float16), tensor(20.0625, dtype=torch.float16))

In [16]:
probs1 = masked_softmax([568, 1364], test_logits)
probs1 = probs1/torch.sum(probs1) # Normalized

In [17]:
probs1.detach().numpy()

array([0.3923, 0.6074], dtype=float16)

In [18]:
0.3738 + 0.6260

0.9998

## Contextuality

In [19]:
# In the CSV, "he first" is always the original direction and then it is reversed in the feminine case

In [20]:
# Probably need to output the tokenized prompt from the function, not the list OR input the tokenized prompt here
ms = MeasurementScenario(observations=['template_1', 'template_2'], 
                         measurements=['he_first', 'she_first'],
                         outcomes=[0,1]) # 1 corresponds to selecting the feminine pronoun

In [21]:
ms.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [22]:
ms.outcome_pairs

[(0, 0), (0, 1), (1, 0), (1, 1)]

In [23]:
def reverse_pronouns(options,
                     measurement, 
                     prime='she_first'):
    
    pronouns = ast.literal_eval(options)
    if measurement==prime:
        return pronouns[::-1]
    else:
        return pronouns

In [25]:
# This is gonna be the function 

for arr_idx, pair in enumerate(ms.context_pairs):
    arr = np.zeros((len(ms.observations), len(ms.outcomes)))
    for pair_idx, oc_idx in enumerate(pair):
        oc_pair = ms.reverse_context_idx.get(oc_idx)
        obs, ctx = oc_pair
        obs_index = obs[-1]
        sent = pairs[obs][0] # the 0 is iterated index
        pnouns = reverse_pronouns(pairs[f"differences_{obs_index}"][0], ctx) 
        prompt = wc.modeling.prompting.get_role_content_prompt(game=False, options=pnouns, sentence=sent)
        logits = mp.get_raw_logits(prompt=prompt).to('cpu')
        # For now, we just use the two tokens 
        tokens = mp.get_token_ids(options=[" "+s for s in ast.literal_eval(pairs[f"differences_{obs_index}"][0])]) # in order [m, f]
        softmax = masked_softmax(list(chain.from_iterable(tokens)), logits)
        probs = softmax/torch.sum(softmax)
        arr[pair_idx] = probs.detach().numpy()
    joint = compute_joint(arr)
    renorm_joint = joint / np.sum(joint)
    ms.scenario[arr_idx] = renorm_joint.reshape(-1)


In [26]:
ms.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0.16134076, 0.23108903, 0.24979205, 0.35777816],
       [0.06046965, 0.33196015, 0.09362071, 0.5139495 ],
       [0.03256746, 0.0466465 , 0.37856535, 0.54222068],
       [0.01220611, 0.06700786, 0.14188425, 0.77890179]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [28]:
np.sum(ms.scenario[0,:])

<xarray.DataArray ()> Size: 8B
array(1.)
Coordinates:
    context_pair  int64 8B 0

In [29]:
ms.incidence_matrix()

<xarray.DataArray (s: 16, t: 16)> Size: 2kB
array([[1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1.],
       [1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1.]])
Coordinates:
  * s        (s) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * t        (t) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15

In [30]:
check_feasibility(ms)

2025-07-25 17:46:37.056 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:107 - Measurement status: 0


(True,
         message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
         success: True
          status: 0
             fun: 0.0
               x: [ 1.221e-02  2.036e-02 ...  9.362e-02  2.175e-01]
             nit: 11
           lower:  residual: [ 1.221e-02  2.036e-02 ...  9.362e-02
                               2.175e-01]
                  marginals: [ 0.000e+00  0.000e+00 ...  0.000e+00
                               0.000e+00]
           upper:  residual: [ 9.878e-01  9.796e-01 ...  9.064e-01
                               7.825e-01]
                  marginals: [ 0.000e+00  0.000e+00 ...  0.000e+00
                               0.000e+00]
           eqlin:  residual: [-2.776e-17  0.000e+00 ...  0.000e+00
                               0.000e+00]
                  marginals: [-0.000e+00 -0.000e+00 ... -0.000e+00
                              -0.000e+00]
         ineqlin:  residual: []
                  marginals: []
  mip_node_count: 0
  mip_dual_bound: 0

In [31]:
contextuality = check_feasibility(ms)

2025-07-25 17:46:39.632 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:107 - Measurement status: 0


In [32]:
contextuality[0]

True

In [42]:
calculate_contextual_fraction_abramsky(ms)

2025-07-25 17:47:13.336 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:170 - Noncontextual fraction: 1.000000
2025-07-25 17:47:13.336 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:171 - Contextual fraction: 0.000000
2025-07-25 17:47:13.336 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:175 - Model is non-contextual (within numerical precision)


1.1102230246251565e-16

In [34]:
mp.get_token_ids(options=[' potato'])

[[39834]]

In [35]:
import torch.nn.functional as F

In [36]:
F.softmax(test_logits, dim=0)[39834]

tensor(0., dtype=torch.float16)

In [37]:
mp.get_token_ids(options=[" "+s for s in ast.literal_eval(pairs[f"differences_{obs_index}"][0])])[1][0]

1364

# Testing the Counterexample

In [8]:
df = pd.read_csv(wc.config.PROCESSED_DATA_DIR / "wino_pairs.tsv", sep="\t")

In [10]:
df

,template_1,differences_1,referent_1,template_2,differences_2,referent_2
0,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
1,The customer met with the technician to get he...,"['his', 'her']",customer,The customer met with the technician to find o...,"['he', 'she']",technician
2,The technician gave the customer feedback on B...,"['his', 'her']",customer,The customer gave the technician feedback on B...,"['his', 'her']",technician
3,The technician informed the customer that BLAN...,"['he', 'she']",customer,The technician informed the customer that BLAN...,"['he', 'she']",technician
4,The technician told the customer that BLANK sh...,"['he', 'she']",customer,The technician told the customer that BLANK wa...,"['he', 'she']",technician
...,...,...,...,...,...,...
56,The technician came out to apologize to the cu...,"['his', 'her']",customer,The technician came out to apologize to the cu...,"['his', 'her']",technician
57,The technician told the customer that BLANK wo...,"['he', 'she']",customer,The technician told the customer that BLANK wo...,"['he', 'she']",technician
58,The technician had to rescue the customer from...,"['he', 'she']",customer,The technician had to rescue the customer from...,"['he', 'she']",technician
59,The technician asked the customer to sign in s...,"['he', 'she']",customer,The technician asked the customer to sign in s...,"['he', 'she']",technician


In [19]:
# Probably need to output the tokenized prompt from the function, not the list OR input the tokenized prompt here
ms1 = MeasurementScenario(observations=['template_1', 'template_2'], 
                         measurements=['he_first', 'she_first'],
                         outcomes=[0,1]) # 1 corresponds to selecting the feminine pronoun

In [62]:
for arr_idx, pair in enumerate(ms1.context_pairs):
    arr = np.zeros((len(ms1.observations), len(ms1.outcomes)))
    for pair_idx, oc_idx in enumerate(pair):
        oc_pair = ms1.reverse_context_idx.get(oc_idx)
        obs, ctx = oc_pair
        obs_index = obs[-1]
        sent = df[obs][60] # the 0 is iterated index
        pnouns = reverse_pronouns(df[f"differences_{obs_index}"][60], ctx) 
        prompt = wc.modeling.prompting.get_role_content_prompt(game=False, options=pnouns, sentence=sent)
        logits = mp.get_raw_logits(prompt=prompt).to('cpu')
        # For now, we just use the two tokens 
        token = mp.get_token_ids(options=[" " + s for s in ast.literal_eval(
                    df[f"differences_{obs_index}"][60])])[1][0]  # in order [m, f]
        softmax = F.softmax(logits, dim=0)
        prob = softmax[token]
        probs = np.array([1-prob, prob])
        arr[pair_idx] = probs
    ms1.scenario[arr_idx] = compute_joint(arr).reshape(-1) # does this work

In [63]:
ms1.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0.90952635, 0.06233126, 0.02652309, 0.00181767],
       [0.89719057, 0.07448912, 0.02616336, 0.00217221],
       [0.80074883, 0.05487657, 0.13551486, 0.00928704],
       [0.78988838, 0.06558037, 0.13367689, 0.0110985 ]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [64]:
check_feasibility(ms1)

2025-07-25 17:36:00.926 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:107 - Measurement status: 2


(False,
        message: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
        success: False
         status: 2
            fun: None
              x: None
            nit: 10
          lower:  residual: None
                 marginals: None
          upper:  residual: None
                 marginals: None
          eqlin:  residual: None
                 marginals: None
        ineqlin:  residual: None
                 marginals: None)

In [65]:
calculate_contextual_fraction_abramsky(ms1)

2025-07-25 17:36:02.260 | WARNING  | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:138 - Probabilities don't sum to 4 (sum = 4.000885057263076)
2025-07-25 17:36:02.260 | WARNING  | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:139 - This may indicate an issue with the probability distribution
2025-07-25 17:36:02.264 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:170 - Noncontextual fraction: 1.000015
2025-07-25 17:36:02.265 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:171 - Contextual fraction: 0.000000
2025-07-25 17:36:02.265 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:175 - Model is non-contextual (within numerical precision)


0.0

# Non-Contextual Examples

In [ ]:
nc_ms = MeasurementScenario(observations=['template_1', 'template_2'], 
                         measurements=['he_first', 'she_first'],
                         outcomes=[0,1]) 

In [ ]:
nc_table = np.array([
    [1.0, 0.0, 0.0, 0.0],  # context (a₀, b₀): Alice gets 0→outputs 0, Bob→0, so P(0,0)=1
    [1.0, 0.0, 0.0, 0.0],  # context (a₀, b₁): Alice gets 0→outputs 0, Bob→0, so P(0,0)=1
    [0.0, 0.0, 1.0, 0.0],  # context (a₁, b₀): Alice gets 1→outputs 1, Bob→0, so P(1,0)=1  
    [0.0, 0.0, 1.0, 0.0]   # context (a₁, b₁): Alice gets 1→outputs 1, Bob→0, so P(1,0)=1
])

In [ ]:
nc_ms.scenario.values = nc_table

In [ ]:
check_feasibility(nc_ms)

In [ ]:
ks_table = np.array([
    [0.5, 0.0, 0.0, 0.5],  # context (σ_x, σ_y): anticorrelated  
    [0.5, 0.0, 0.0, 0.5],  # context (σ_x, σ_z): anticorrelated
    [0.5, 0.0, 0.0, 0.5],  # context (σ_y, σ_x): anticorrelated  
    [0.5, 0.0, 0.0, 0.5]   # context (σ_y, σ_z): anticorrelated
])

In [ ]:
nc_ms.scenario.values = ks_table
check_feasibility(nc_ms)

# Contextuality by Default

In [9]:
def message(sys_prompt, user_prompt, ass_prompt):
    message = [
        {
            "role": "system",
            "content": sys_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": ass_prompt
        }
    ]

    return message

In [10]:
def no_game_sim_prompt(option_sets: list[list[str]],
                       sentences: list[str]):

    """
    Outputs a zero-shot user prompt which needs to be fit into a template.

    :param options: A LIST of pronoun options. MUST be ordered to measure contextuality.
    :param sentence: Orderedlist of two sentences, each with a BLANK
    :return: user prompt
    """

    sent1, sent2 = sentences
    sent1.replace('BLANK', 'BLANK1')
    sent2.replace('BLANK', 'BLANK2')
    

    SYSTEM_PROMPT = ("Below you will find a passage in *bold* which contains precisely one instances of "
                     "the term BLANK1, and one instance of the term BLANK2. "
                     "Your task is to replace BLANK1 and BLANK2 with one of the options provided for each. "
                     "The tasks are designed to be unambiguous, so please provide only one token for each blank and "
                     "do not reorder the data. Do not repeat the sentence.")

    USER_PROMPT = (f"Given this passage: *{sent1} {sent2}*\n" \
                   f"Replace BLANK1 with one of the options: {option_sets[0]}. " \
                   f"Replace BLANK2 with one of the options: {option_sets[1]}. "
                   "Respond only in the following format {'BLANK1': '<text>', 'BLANK2': '<text>'}"
                  ) 
    
    ASSISTANT_PROMPT = "{'BLANK1':"


    return SYSTEM_PROMPT, USER_PROMPT, ASSISTANT_PROMPT

In [11]:
options_sets = [ast.literal_eval(df.differences_1[0]), ast.literal_eval(df.differences_2[0])]
sentences = [df.template_1[0], df.template_2[0]]

prompt = message(*no_game_sim_prompt(options_sets, sentences))

inputs = mp.tokenizer.apply_chat_template(prompt, return_tensors="pt", continue_final_message=True).to("cuda:0")

In [12]:
outputs = mp.model.generate(inputs, 
                            max_new_tokens = 12,
                            output_scores=True, 
                            return_dict_in_generate=True, 
                            output_hidden_states=True, 
                            do_sample = True, 
                            temperature = 0.5, 
                            pad_token_id=mp.tokenizer.eos_token_id,
                            #eos_token_id=mp.tokenizer.eos_token_id,
                            top_k=2)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [13]:
mp.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

"system\n\nCutting Knowledge Date: December 2023\nToday Date: 28 Jul 2025\n\nBelow you will find a passage in *bold* which contains precisely one instances of the term BLANK1, and one instance of the term BLANK2. Your task is to replace BLANK1 and BLANK2 with one of the options provided for each. The tasks are designed to be unambiguous, so please provide only one token for each blank and do not reorder the data. Do not repeat the sentence.user\n\nGiven this passage: *The technician told the customer that BLANK could pay with cash. The technician told the customer that BLANK had completed the repair.*\nReplace BLANK1 with one of the options: ['he','she']. Replace BLANK2 with one of the options: ['he','she']. Respond only in the following format {'BLANK1': '<text>', 'BLANK2': '<text>'}assistant\n\n{'BLANK1': 'he', 'BLANK2':'she'}"

In [14]:
outputs.scores

(tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'))

In [15]:
input_length = inputs.shape[1]
generated_tokens = outputs.sequences[0][input_length:]
token_probs = []

for token_id, score in zip(generated_tokens, outputs.scores):
    print(token_id, score)
    if score.dim() == 2:
        score = score[0]  # from shape [1, vocab_size] to [vocab_size]
    
    probs = F.softmax(score, dim=-1)
    prob = probs[token_id].item()
    token = mp.tokenizer.convert_ids_to_tokens(int(token_id))
    token_probs.append((token, prob))

# Display
for token, prob in token_probs:
    print(f"{token}\t{prob:.4f}")

tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(383, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(518, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(9574, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(16395, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(17, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(1232, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(32158, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tenso

In [16]:
raw_input_str1 = "The technician told the customer that "
input1 = mp.tokenizer(raw_input_str1, return_tensors="pt").to('cuda:0')
outputs1 = mp.model(**input1)
outputs1.logits[0, -1]

tensor([1.8438, 3.3379, 3.6289,  ..., 0.7388, 0.7397, 0.7393], device='cuda:0',
       dtype=torch.float16, grad_fn=<SelectBackward0>)

In [17]:
# marginals for the case where it's Q2 have to be calculated based on the conditionals (which would have to be over every word in the vocab)

In [18]:
raw_input_str1 = "The technician told the customer that "
input1 = mp.tokenizer(raw_input_str1, return_tensors="pt").to('cuda:0')
outputs1 = mp.model(**input1)
outputs1.logits[0, -1]

tensor([1.8438, 3.3379, 3.6289,  ..., 0.7388, 0.7397, 0.7393], device='cuda:0',
       dtype=torch.float16, grad_fn=<SelectBackward0>)

# Repeated Testing on CbD

In [19]:
decoded_outputs = []

for idx in df.index[:10]:
    options_sets_idx = [ast.literal_eval(df.differences_1[idx]), ast.literal_eval(df.differences_2[idx])]
    sentences_idx = [df.template_1[idx], df.template_2[idx]]
    
    prompt_idx = message(*no_game_sim_prompt(options_sets_idx, sentences_idx))
    
    inputs_idx = mp.tokenizer.apply_chat_template(prompt_idx, return_tensors="pt", continue_final_message=True).to("cuda:0")

    outputs_idx = mp.model.generate(inputs_idx, 
                                    max_new_tokens = 12,
                                    output_scores=True, 
                                    return_dict_in_generate=True, 
                                    output_hidden_states=True, 
                                    do_sample = True, 
                                    temperature = 0.5, 
                                    pad_token_id=mp.tokenizer.eos_token_id,
                                    #eos_token_id=mp.tokenizer.eos_token_id,
                                    top_k=2)

    input_length_idx = inputs_idx.shape[1]
    print(mp.tokenizer.decode(outputs_idx.sequences[0], skip_special_tokens=True))
    decoded_outputs.append(mp.tokenizer.decode(outputs_idx.sequences[0][input_length_idx - 5:], skip_special_tokens=True))
    

system

Cutting Knowledge Date: December 2023
Today Date: 28 Jul 2025

Below you will find a passage in *bold* which contains precisely one instances of the term BLANK1, and one instance of the term BLANK2. Your task is to replace BLANK1 and BLANK2 with one of the options provided for each. The tasks are designed to be unambiguous, so please provide only one token for each blank and do not reorder the data. Do not repeat the sentence.user

Given this passage: *The technician told the customer that BLANK could pay with cash. The technician told the customer that BLANK had completed the repair.*
Replace BLANK1 with one of the options: ['he','she']. Replace BLANK2 with one of the options: ['he','she']. Respond only in the following format {'BLANK1': '<text>', 'BLANK2': '<text>'}assistant

{'BLANK1': 'he', 'BLANK2':'she'}
system

Cutting Knowledge Date: December 2023
Today Date: 28 Jul 2025

Below you will find a passage in *bold* which contains precisely one instances of the term BLANK1, 

In [20]:
mp.tokenizer.decode(outputs_idx.sequences[0][input_length_idx - 5:], skip_special_tokens=True)

"{'BLANK1': 'his', 'BLANK2':'she'}"

In [21]:
mp.tokenizer(' she'), mp.tokenizer.convert_ids_to_tokens([1364])

({'input_ids': [128000, 1364], 'attention_mask': [1, 1]}, ['Ġshe'])

In [22]:
mp.tokenizer(' he'), mp.tokenizer.convert_ids_to_tokens([568])

({'input_ids': [128000, 568], 'attention_mask': [1, 1]}, ['Ġhe'])

In [23]:
token_probs = []
for token_id, score in zip(outputs_idx.sequences[0][input_length_idx:], outputs_idx.scores):
    print(token_id, score)
    if score.dim() == 2:
        score = score[0]  # from shape [1, vocab_size] to [vocab_size]
    
    probs = F.softmax(score, dim=-1)
    prob = probs[token_id].item()
    token = mp.tokenizer.convert_ids_to_tokens(int(token_id))
    token_probs.append((token, prob))

# Display
for token, prob in token_probs:
    print(f"{token}\t{prob:.4f}")

tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(26301, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(518, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(9574, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(16395, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(17, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(1232, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(364, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
tensor(32158, device='cuda:0') tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')
ten

In [24]:
mp.tokenizer.convert_ids_to_tokens([9574])

['BL']

In [25]:
mp.tokenizer('BLANK2').input_ids[1]

9574

In [26]:
mp.tokenizer(['her'])

{'input_ids': [[128000, 1964]], 'attention_mask': [[1, 1]]}

In [27]:
type(outputs_idx.sequences[0][input_length_idx:])

torch.Tensor

In [28]:
outputs.scores[0]

tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')

In [29]:
test_list = ['his', 'her']
test_tok = mp.tokenizer(test_list)
possible_test = torch.tensor([t[1] for t in test_tok.input_ids]).to('cuda:0')
mask = (outputs_idx.sequences[0][input_length_idx:][..., None] == possible_test).any(-1)
indices = torch.nonzero(mask, as_tuple=False)
indices

tensor([[1]], device='cuda:0')

In [30]:
outputs_idx.scores[indices.item()]

tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')

In [31]:
mp.tokenizer.decode(outputs_idx.sequences[0], skip_special_tokens=True)

"system\n\nCutting Knowledge Date: December 2023\nToday Date: 28 Jul 2025\n\nBelow you will find a passage in *bold* which contains precisely one instances of the term BLANK1, and one instance of the term BLANK2. Your task is to replace BLANK1 and BLANK2 with one of the options provided for each. The tasks are designed to be unambiguous, so please provide only one token for each blank and do not reorder the data. Do not repeat the sentence.user\n\nGiven this passage: *The technician called to inform the customer that BLANK car would be ready in the morning. The technician called to inform the customer that BLANK had completed the repair.*\nReplace BLANK1 with one of the options: ['his', 'her']. Replace BLANK2 with one of the options: ['he','she']. Respond only in the following format {'BLANK1': '<text>', 'BLANK2': '<text>'}assistant\n\n{'BLANK1': 'his', 'BLANK2':'she'}"

In [35]:
def find_pronouns(pronouns: list,
                  generated_sequence: torch.Tensor,
                  output_scores: tuple[torch.Tensor] | None = None,
                  logits: bool = True) -> torch.Tensor | tuple[torch.Tensor]:

    """
    :param pronouns: list of pronouns searching for
    :param generated_sequence: only the tokens generated
    :param scores: output scores for generated tokens, only needed if logits = True
    :param logits: whether to return logits tensors
    """
    
    
    trial_strings = pronouns + [' '+p for p in pronouns]
    token_tensor = mp.tokenizer(trial_strings + pronouns)
    possible_tokens = torch.tensor([t[1] for t in token_tensor.input_ids]).to('cuda:0')
    mask = (generated_sequence[..., None] == possible_tokens).any(-1)
    indices = torch.nonzero(mask, as_tuple=False)

    if logits:
        if output_scores == None:
            raise TypeError("Output scores required to output logits.")
        else:
            output_logits = output_scores[indices.item()]
            return indices, output_logits

    else:
        return indices
        
    

In [36]:
sep_token = mp.tokenizer('BLANK2').input_ids[1]

def pronoun_logits(pronouns_list: list[list[str]],
                   generated_sequence: torch.Tensor,
                   output_scores: tuple[torch.Tensor] | None = None
                  ) -> torch.Tensor | tuple[torch.Tensor]:

    token_log_dict = {g.cpu().item(): arr for g, arr in zip(generated_sequence, output_scores)}
    idx = list(token_log_dict.keys()).index(sep_token)

    first_idx, first_logits = find_pronouns(pronouns_list[0], generated_sequence[:idx], output_scores[:idx], logits=True)
    second_idx, second_logits = find_pronouns(pronouns_list[1], generated_sequence[idx+1:], 
                                  output_scores[idx+1:], logits=True)

    first_token = mp.tokenizer.convert_ids_to_tokens([generated_sequence[:idx][first_idx.item()]])
    second_token = mp.tokenizer.convert_ids_to_tokens([generated_sequence[idx+1:][second_idx.item()]])

    return (first_token, first_logits, second_token, second_logits)
    

In [37]:
a,b,c,d = pronoun_logits([['his', 'her'], ['he', 'she']], 
               outputs_idx.sequences[0][input_length_idx:],
               outputs_idx.scores)

In [38]:
mp.tokenizer.convert_ids_to_tokens(383)

'he'

In [42]:
masked_softmax([383], d[0]).cpu()

tensor([1.])

In [55]:
test_dict = {g.item(): arr for g, arr in zip(outputs_idx.sequences[0][input_length_idx:], outputs_idx.scores)}
test_dict

{364: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 26301: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 518: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 9574: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 16395: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 17: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 1232: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 383: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 8439: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 128009: tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')}

In [ ]:
list(test_dict.keys()).index(sep_token)

In [248]:
find_pronouns(test_list, outputs_idx.sequences[0][input_length_idx:], outputs_idx.scores)[1][0][26301]

tensor(44.4375, device='cuda:0')

In [250]:
l = ['a', 'b']
for n in range(2):
    print(l[:n])

[]
['a']


In [255]:
[ast.literal_eval(d) for d in decoded_outputs][0]

{'BLANK1': 'He', 'BLANK2': 'She'}

In [256]:
from itertools import permutations

In [260]:
test_dict ={1: 'a', 2: 'b'}
for i, test_perm in enumerate(permutations(test_dict.keys())):
    print(i, test_perm)

0 (1, 2)
1 (2, 1)
